# 02 — Phase 1: Image Deepfake Detection

**AI Media Authenticity / Deepfake Detection Platform**

This notebook trains and evaluates the image classifier (real face vs. AI-generated/fake face).

It reuses the exact 4,000-image subset and the 80/10/10 train/val/test split that
`01_Phase_0_Dataset_Preparation.ipynb` already built and saved to CSV — this notebook
does **not** re-run dataset discovery or change Phase 0 in any way, it just loads the
CSVs Phase 0 already produced:

- 4,000 images total (2,000 real + 2,000 fake)
- 3,200 training / 400 validation / 400 test

What this notebook does, step by step:

1. Load the Phase 0 train/val/test CSVs
2. Build a lazy PyTorch `Dataset`/`DataLoader` (images are opened one at a time, never all at once)
3. Resize every image to 224x224 and normalize it
4. Load a small pretrained Vision Transformer and fine-tune it (transfer learning, not training from scratch)
5. Train for a few epochs, saving the best model as validation accuracy improves
6. Evaluate on the held-out test set: accuracy, precision, recall, F1, confusion matrix
7. Plot training/validation loss and accuracy curves
8. Show a handful of test images with their actual label, predicted label, and confidence
9. Save the evaluation results to a JSON file

**Batch size** is picked automatically depending on where this runs:
- Local laptop, no GPU: batch size 2
- Local laptop, GPU with little free memory: batch size 2
- Local laptop, GPU with enough free memory: batch size 4
- Kaggle with a GPU: batch size 16 (Kaggle's GPUs have more memory to spare)

This keeps memory use safe for an 8 GB RAM laptop while still training a bit faster
on Kaggle.


## 1. Project path bootstrap

Same approach as Notebook 01 — finds the `ml/` folder regardless of where Jupyter was launched from.

In [ ]:
import sys
from pathlib import Path


def find_ml_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(6):
        if (current / "src" / "config.py").exists():
            return current
        current = current.parent
    raise FileNotFoundError(
        "Could not find the 'ml/' project root (looked for src/config.py up to "
        "6 parent folders above the current directory). If you're on Kaggle, "
        "make sure the 'ml' folder is available under /kaggle/working."
    )


ML_ROOT = find_ml_root(Path.cwd())
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

print(f"ML project root resolved to: {ML_ROOT}")


## 2. Imports

In [ ]:
import json

import pandas as pd
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

from src import config
from src.utils import set_seed, get_device, choose_batch_size
from src.data import build_image_dataloaders
from src.models import build_image_model
from src.training import train_model
from src.evaluation import (
    get_predictions,
    compute_metrics,
    print_classification_report,
    plot_confusion_matrix,
    plot_training_curves,
)

set_seed(config.RANDOM_STATE)
print("Imports OK. Random seed set to", config.RANDOM_STATE)


## 3. Configuration

Most of these values come straight from `src/config.py` (the same file Phase 0 uses), so everything in the project agrees on what "224x224" and "a few epochs" actually mean. Batch size is the one exception - it's decided here based on the hardware we're actually running on.

In [ ]:
DEVICE = get_device()
BATCH_SIZE = choose_batch_size(DEVICE)

IMAGE_SIZE = config.IMAGE_SIZE
NUM_WORKERS = config.NUM_WORKERS
EPOCHS = config.NUM_EPOCHS
LEARNING_RATE = config.LEARNING_RATE
BACKBONE_NAME = config.VIT_BACKBONE

print(f"Device        : {DEVICE}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Image size    : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Epochs        : {EPOCHS}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"ViT backbone  : {BACKBONE_NAME}")


## 4. Load the Phase 0 train/val/test CSVs

These files were created by `01_Phase_0_Dataset_Preparation.ipynb`. If they're missing, that notebook needs to be run first — we don't try to rebuild them here, since Phase 0 is not something this notebook should touch.

In [ ]:
for csv_path in [config.IMAGE_TRAIN_CSV, config.IMAGE_VALID_CSV, config.IMAGE_TEST_CSV]:
    if not Path(csv_path).exists():
        raise FileNotFoundError(
            f"{csv_path} does not exist yet. Run "
            "01_Phase_0_Dataset_Preparation.ipynb first - it creates the "
            "train/val/test split this notebook needs."
        )

train_df = pd.read_csv(config.IMAGE_TRAIN_CSV)
val_df = pd.read_csv(config.IMAGE_VALID_CSV)
test_df = pd.read_csv(config.IMAGE_TEST_CSV)

print(f"Train images: {len(train_df)}")
print(f"Val images  : {len(val_df)}")
print(f"Test images : {len(test_df)}")

print("\nTrain label balance:")
print(train_df['label'].value_counts())


## 5. Build the DataLoaders

Each `Dataset` only reads its CSV of filepaths up front (a tiny amount of memory) - the actual image files are opened one batch at a time by the DataLoader while training runs. Nothing here loads all 3,200 training images into memory at once.

In [ ]:
train_loader, val_loader, test_loader = build_image_dataloaders(
    train_csv=config.IMAGE_TRAIN_CSV,
    val_csv=config.IMAGE_VALID_CSV,
    test_csv=config.IMAGE_TEST_CSV,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

# Quick sanity check: pull just one batch and look at its shape.
sample_images, sample_labels = next(iter(train_loader))
print(f"One batch of images has shape: {tuple(sample_images.shape)}")
print(f"One batch of labels has shape: {tuple(sample_labels.shape)}")
print(f"Classes (in label-index order): {config.IMAGE_CLASSES}")


## 6. Build the model (transfer learning)

We start from a ViT that's already pretrained on ImageNet and just replace its last layer with a new one for our 2 classes (real / fake). This is much faster and needs far less data than training a ViT from nothing.

In [ ]:
model = build_image_model(num_classes=len(config.IMAGE_CLASSES), backbone_name=BACKBONE_NAME, pretrained=True)

num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {BACKBONE_NAME}")
print(f"Total parameters    : {num_params:,}")
print(f"Trainable parameters: {num_trainable:,}")


## 7. Train

Saves the model to `ml/checkpoints/image_vit_best.pt` every time validation accuracy improves, so we keep the best version even if it starts overfitting in a later epoch.

In [ ]:
CHECKPOINT_PATH = config.CHECKPOINTS_DIR / "image_vit_best.pt"

history, best_val_accuracy = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    checkpoint_path=CHECKPOINT_PATH,
)

print(f"\nBest validation accuracy reached: {best_val_accuracy:.4f}")
print(f"Best model saved to: {CHECKPOINT_PATH}")


## 8. Plot training curves

In [ ]:
curves_path = config.REPORTS_DIR / "image_training_curves.png"
plot_training_curves(history, save_path=curves_path)


## 9. Load the best model and evaluate on the test set

We reload the best checkpoint from disk (rather than just using whatever the model looks like after the last epoch) so the numbers below reflect the best version we saved, not necessarily the final one.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.to(DEVICE)

y_true, y_pred, y_confidence = get_predictions(model, test_loader, DEVICE)

test_metrics = compute_metrics(y_true, y_pred)
print("Test set metrics:")
for name, value in test_metrics.items():
    print(f"  {name}: {value:.4f}")

print()
print_classification_report(y_true, y_pred, class_names=config.IMAGE_CLASSES)


## 10. Confusion matrix

In [ ]:
cm_path = config.REPORTS_DIR / "image_confusion_matrix.png"
confusion = plot_confusion_matrix(y_true, y_pred, class_names=config.IMAGE_CLASSES, save_path=cm_path)


## 11. A few example test predictions

Green title = the model got it right. Red title = the model got it wrong.

In [ ]:
num_examples = 8
rng = np.random.RandomState(config.RANDOM_STATE)
example_positions = rng.choice(len(test_df), size=num_examples, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for ax, position in zip(axes.flatten(), example_positions):
    row = test_df.iloc[position]
    image = Image.open(row["filepath"]).convert("RGB")

    actual_label = config.IMAGE_CLASSES[y_true[position]]
    predicted_label = config.IMAGE_CLASSES[y_pred[position]]
    confidence = y_confidence[position]
    correct = actual_label == predicted_label

    ax.imshow(image)
    ax.axis("off")
    ax.set_title(
        f"Actual: {actual_label}\nPredicted: {predicted_label} ({confidence:.2f})",
        color="green" if correct else "red",
        fontsize=9,
    )

fig.tight_layout()
predictions_image_path = config.REPORTS_DIR / "image_sample_predictions.png"
fig.savefig(predictions_image_path)
print(f"Saved sample predictions image to {predictions_image_path}")
plt.show()


## 12. Save the evaluation results

In [ ]:
results = {
    "model_name": BACKBONE_NAME,
    "num_train_images": len(train_df),
    "num_val_images": len(val_df),
    "num_test_images": len(test_df),
    "best_val_accuracy": best_val_accuracy,
    "test_accuracy": test_metrics["accuracy"],
    "test_precision": test_metrics["precision"],
    "test_recall": test_metrics["recall"],
    "test_f1_score": test_metrics["f1_score"],
    "checkpoint_path": str(CHECKPOINT_PATH),
}

results_path = config.REPORTS_DIR / "image_evaluation_results.json"
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved evaluation results to {results_path}")


## 13. Final summary

In [ ]:
print("=" * 60)
print("PHASE 1 - IMAGE DETECTION SUMMARY")
print("=" * 60)
print(f"""
Model name              : {BACKBONE_NAME}
Training images         : {len(train_df)}
Validation images       : {len(val_df)}
Test images             : {len(test_df)}
Best validation accuracy: {best_val_accuracy:.4f}
Test accuracy           : {test_metrics['accuracy']:.4f}
Test precision          : {test_metrics['precision']:.4f}
Test recall             : {test_metrics['recall']:.4f}
Test F1-score           : {test_metrics['f1_score']:.4f}
Saved model location    : {CHECKPOINT_PATH}
""")

print("PHASE 1 IMAGE DETECTION COMPLETED")
